In [1]:
# ============================================================
# NOTEBOOK 02A
# RECOVER MISSING S1HAND AND S2HAND RASTERS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import hashlib
import json
import os
import time

import pandas as pd
import rasterio


print("=" * 80)
print("NOTEBOOK 02A: RECOVER MISSING S1HAND AND S2HAND RASTERS")
print("=" * 80)


# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TABLES_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

RAW_HANDLABELED_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "sen1floods11"
    / "hand_labeled"
)

S1_OUTPUT_DIR = (
    RAW_HANDLABELED_DIR
    / "S1Hand"
)

S2_OUTPUT_DIR = (
    RAW_HANDLABELED_DIR
    / "S2Hand"
)

RECOVERY_OUTPUT_DIR = (
    TABLES_DIR
    / "recovery"
)

VLM_SCENE_MANIFEST_CSV = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vlm_dataset"
    / "master"
    / "scene_multimodal_manifest.csv"
)

SCENE_QUALITY_MANIFEST_CSV = (
    TABLES_DIR
    / "roadflood_vlm_scene_quality_manifest.csv"
)

PROTOTYPE_MANIFEST_CSV = (
    TABLES_DIR
    / "roadflood_vlm_prototype_download_manifest.csv"
)


for directory in [
    S1_OUTPUT_DIR,
    S2_OUTPUT_DIR,
    RECOVERY_OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print("\nDIRECTORIES")
print("-" * 80)

print(
    f"Project root          : {PROJECT_ROOT}"
)

print(
    f"S1 output directory   : {S1_OUTPUT_DIR}"
)

print(
    f"S2 output directory   : {S2_OUTPUT_DIR}"
)

print(
    f"Recovery output       : {RECOVERY_OUTPUT_DIR}"
)


# ------------------------------------------------------------
# 2. Validate source manifests
# ------------------------------------------------------------

SOURCE_FILES = {
    "vlm_scene_manifest": (
        VLM_SCENE_MANIFEST_CSV
    ),
    "scene_quality_manifest": (
        SCENE_QUALITY_MANIFEST_CSV
    ),
    "prototype_manifest": (
        PROTOTYPE_MANIFEST_CSV
    ),
}


print("\nSOURCE FILE VALIDATION")
print("-" * 80)

for source_name, source_path in (
    SOURCE_FILES.items()
):
    print(
        f"{source_name:<26}: "
        f"{source_path.exists()} | "
        f"{source_path}"
    )


missing_source_files = [
    str(
        source_path
    )
    for source_path in (
        SOURCE_FILES.values()
    )
    if not source_path.exists()
]


if missing_source_files:
    raise FileNotFoundError(
        "Required source files are missing:\n"
        + "\n".join(
            missing_source_files
        )
    )


# ------------------------------------------------------------
# 3. Load manifests
# ------------------------------------------------------------

vlm_scene_manifest_df = pd.read_csv(
    VLM_SCENE_MANIFEST_CSV,
    keep_default_na=False,
    low_memory=False,
)

scene_quality_manifest_df = pd.read_csv(
    SCENE_QUALITY_MANIFEST_CSV,
    keep_default_na=False,
    low_memory=False,
)

prototype_manifest_df = pd.read_csv(
    PROTOTYPE_MANIFEST_CSV,
    keep_default_na=False,
    low_memory=False,
)


required_scene_ids = sorted(
    vlm_scene_manifest_df[
        "scene_id"
    ]
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)


print("\nLOADED MANIFESTS")
print("-" * 80)

print(
    f"Required VLM scenes         : "
    f"{len(required_scene_ids):,}"
)

print(
    f"Scene-quality records       : "
    f"{len(scene_quality_manifest_df):,}"
)

print(
    f"Prototype records           : "
    f"{len(prototype_manifest_df):,}"
)


if len(required_scene_ids) == 0:
    raise ValueError(
        "No required VLM scenes were found in the manifest."
    )


# ------------------------------------------------------------
# 4. Verify the remote naming convention from prototypes
# ------------------------------------------------------------

required_prototype_components = {
    "S1Hand",
    "S2Hand",
    "LabelHand",
    "JRCWaterHand",
}


prototype_components = set(
    prototype_manifest_df[
        "component"
    ]
    .astype(str)
    .str.strip()
    .unique()
)


missing_prototype_components = (
    required_prototype_components
    - prototype_components
)


print("\nPROTOTYPE COMPONENT VALIDATION")
print("-" * 80)

print(
    f"Prototype components observed : "
    f"{sorted(prototype_components)}"
)

print(
    f"Missing required components    : "
    f"{sorted(missing_prototype_components)}"
)


if missing_prototype_components:
    raise ValueError(
        "The prototype manifest does not contain all four "
        "expected SEN1Floods11 components."
    )


# Confirm prototype object paths use the expected structure.

prototype_path_validation = {}

for component in [
    "S1Hand",
    "S2Hand",
]:
    component_df = (
        prototype_manifest_df[
            prototype_manifest_df[
                "component"
            ].astype(str).eq(
                component
            )
        ]
    )

    expected_fragment = (
        f"/HandLabeled/{component}/"
    )

    prototype_path_validation[
        component
    ] = (
        component_df[
            "object_name"
        ]
        .astype(str)
        .str.contains(
            expected_fragment,
            regex=False,
        )
        .all()
    )


print("\nREMOTE PATH-CONVENTION VALIDATION")
print("-" * 80)

for component, passed in (
    prototype_path_validation.items()
):
    print(
        f"{component:<12}: "
        f"{bool(passed)}"
    )


if not all(
    bool(
        result
    )
    for result in (
        prototype_path_validation.values()
    )
):
    raise ValueError(
        "Prototype object paths do not follow the expected "
        "HandLabeled component-directory structure."
    )


# ------------------------------------------------------------
# 5. Select one trusted source row per required scene
# ------------------------------------------------------------

scene_quality_manifest_df[
    "scene_id"
] = (
    scene_quality_manifest_df[
        "scene_id"
    ]
    .astype(str)
    .str.strip()
)


selected_source_df = (
    scene_quality_manifest_df[
        scene_quality_manifest_df[
            "scene_id"
        ].isin(
            required_scene_ids
        )
        & scene_quality_manifest_df[
            "component"
        ].astype(str).eq(
            "LabelHand"
        )
    ]
    .copy()
)


selected_source_df = (
    selected_source_df
    .sort_values(
        [
            "scene_id",
            "object_name",
        ]
    )
    .drop_duplicates(
        subset=[
            "scene_id",
        ],
        keep="first",
    )
    .reset_index(
        drop=True
    )
)


print("\nSELECTED-SCENE SOURCE VALIDATION")
print("-" * 80)

print(
    f"Required scenes              : "
    f"{len(required_scene_ids):,}"
)

print(
    f"Label source rows recovered  : "
    f"{len(selected_source_df):,}"
)

print(
    f"Unique source scenes         : "
    f"{selected_source_df['scene_id'].nunique():,}"
)


missing_source_scene_ids = sorted(
    set(
        required_scene_ids
    )
    - set(
        selected_source_df[
            "scene_id"
        ]
    )
)


print(
    f"Missing source scenes        : "
    f"{missing_source_scene_ids}"
)


if missing_source_scene_ids:
    raise ValueError(
        "The scene-quality manifest is missing LabelHand "
        "source rows for these selected scenes: "
        f"{missing_source_scene_ids}"
    )


# ------------------------------------------------------------
# 6. Construct missing S1Hand and S2Hand records
# ------------------------------------------------------------

def replace_component_in_object_name(
    object_name,
    scene_id,
    source_component,
    target_component,
):
    """
    Convert a known component object path into the equivalent
    object path for another SEN1Floods11 component.
    """

    object_name = str(
        object_name
    ).strip()

    source_directory = (
        f"/{source_component}/"
    )

    target_directory = (
        f"/{target_component}/"
    )

    if source_directory not in object_name:
        raise ValueError(
            f"Expected source directory {source_directory} "
            f"was not found in object path: {object_name}"
        )

    target_object_name = object_name.replace(
        source_directory,
        target_directory,
        1,
    )

    source_filename = (
        f"{scene_id}_{source_component}.tif"
    )

    target_filename = (
        f"{scene_id}_{target_component}.tif"
    )

    if target_object_name.endswith(
        source_filename
    ):
        target_object_name = (
            target_object_name[
                :-len(
                    source_filename
                )
            ]
            + target_filename
        )

    else:
        target_object_name = (
            target_object_name
            .rsplit(
                "/",
                1,
            )[
                0
            ]
            + "/"
            + target_filename
        )

    return target_object_name


def object_name_to_public_url(
    object_name,
):
    """
    Convert a SEN1Floods11 object name to its public GCS URL.
    """

    return (
        "https://storage.googleapis.com/"
        "sen1floods11/"
        + str(
            object_name
        ).lstrip(
            "/"
        )
    )


recovery_manifest_records = []

for _, source_row in (
    selected_source_df.iterrows()
):
    scene_id = str(
        source_row[
            "scene_id"
        ]
    ).strip()

    for target_component, output_dir in [
        (
            "S1Hand",
            S1_OUTPUT_DIR,
        ),
        (
            "S2Hand",
            S2_OUTPUT_DIR,
        ),
    ]:
        filename = (
            f"{scene_id}_{target_component}.tif"
        )

        object_name = (
            replace_component_in_object_name(
                object_name=source_row[
                    "object_name"
                ],
                scene_id=scene_id,
                source_component="LabelHand",
                target_component=target_component,
            )
        )

        public_url = (
            object_name_to_public_url(
                object_name
            )
        )

        local_path = (
            output_dir
            / filename
        )

        recovery_manifest_records.append(
            {
                "scene_id": (
                    scene_id
                ),
                "country_prefix": str(
                    source_row[
                        "country_prefix"
                    ]
                ),
                "split": str(
                    source_row[
                        "split"
                    ]
                ),
                "component": (
                    target_component
                ),
                "filename": (
                    filename
                ),
                "object_name": (
                    object_name
                ),
                "public_url": (
                    public_url
                ),
                "local_path": str(
                    local_path
                ),
            }
        )


recovery_manifest_df = pd.DataFrame(
    recovery_manifest_records
)


print("\nRECOVERY MANIFEST CREATED")
print("-" * 80)

print(
    f"Recovery records   : "
    f"{len(recovery_manifest_df):,}"
)

print(
    f"Recovery scenes    : "
    f"{recovery_manifest_df['scene_id'].nunique():,}"
)

print(
    f"S1Hand records     : "
    f"{(recovery_manifest_df['component'] == 'S1Hand').sum():,}"
)

print(
    f"S2Hand records     : "
    f"{(recovery_manifest_df['component'] == 'S2Hand').sum():,}"
)


if len(recovery_manifest_df) != 2 * len(required_scene_ids):
    raise ValueError(
        "Expected 50 recovery records: "
        f"{len(required_scene_ids)} S1Hand plus "
        f"{len(required_scene_ids)} S2Hand."
    )


component_scene_counts = (
    recovery_manifest_df
    .groupby(
        "component"
    )[
        "scene_id"
    ]
    .nunique()
)


if not (
    component_scene_counts.get(
        "S1Hand",
        0,
    )
    == len(required_scene_ids)
    and component_scene_counts.get(
        "S2Hand",
        0,
    )
    == len(required_scene_ids)
):
    raise ValueError(
        "Each recovery component must contain all required scenes."
    )


# ------------------------------------------------------------
# 7. File-integrity helpers
# ------------------------------------------------------------

def calculate_md5(
    file_path,
    chunk_size=1024 * 1024,
):
    """
    Calculate the MD5 hash of a local file.
    """

    md5_hash = hashlib.md5()

    with open(
        file_path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            md5_hash.update(
                chunk
            )

    return md5_hash.hexdigest()


def validate_geotiff(
    file_path,
):
    """
    Validate that a downloaded file can be opened as a GeoTIFF.
    """

    validation = {
        "raster_valid": False,
        "width": None,
        "height": None,
        "band_count": None,
        "dtype": None,
        "crs": None,
        "validation_error": "",
    }

    try:
        with rasterio.open(
            file_path
        ) as dataset:
            validation.update(
                {
                    "raster_valid": bool(
                        dataset.width > 0
                        and dataset.height > 0
                        and dataset.count > 0
                    ),
                    "width": int(
                        dataset.width
                    ),
                    "height": int(
                        dataset.height
                    ),
                    "band_count": int(
                        dataset.count
                    ),
                    "dtype": ",".join(
                        dataset.dtypes
                    ),
                    "crs": (
                        str(
                            dataset.crs
                        )
                        if dataset.crs
                        else ""
                    ),
                }
            )

    except Exception as error:
        validation[
            "validation_error"
        ] = (
            f"{type(error).__name__}: {error}"
        )

    return validation


def existing_file_is_valid(
    file_path,
):
    """
    Check whether an existing local file is a usable raster.
    """

    file_path = Path(
        file_path
    )

    if not file_path.exists():
        return False

    if file_path.stat().st_size <= 0:
        return False

    validation = validate_geotiff(
        file_path
    )

    return bool(
        validation[
            "raster_valid"
        ]
    )


# ------------------------------------------------------------
# 8. Robust downloader
# ------------------------------------------------------------

def download_file(
    url,
    destination_path,
    maximum_attempts=4,
    timeout_seconds=180,
    chunk_size=1024 * 1024,
):
    """
    Download a file using a temporary partial file and retries.
    """

    destination_path = Path(
        destination_path
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = destination_path.with_suffix(
        destination_path.suffix
        + ".part"
    )

    last_error = ""

    for attempt_number in range(
        1,
        maximum_attempts + 1,
    ):
        try:
            if temporary_path.exists():
                temporary_path.unlink()

            request = Request(
                url,
                headers={
                    "User-Agent": (
                        "Mozilla/5.0 "
                        "RoadFlood-VLM-Research-Pipeline/1.0"
                    )
                },
            )

            with urlopen(
                request,
                timeout=timeout_seconds,
            ) as response:
                response_status = getattr(
                    response,
                    "status",
                    200,
                )

                if response_status != 200:
                    raise RuntimeError(
                        "Unexpected HTTP status "
                        f"{response_status}"
                    )

                remote_size_header = (
                    response.headers.get(
                        "Content-Length"
                    )
                )

                remote_size_bytes = (
                    int(
                        remote_size_header
                    )
                    if remote_size_header
                    else None
                )

                bytes_written = 0

                with open(
                    temporary_path,
                    "wb",
                ) as output_file:
                    while True:
                        chunk = response.read(
                            chunk_size
                        )

                        if not chunk:
                            break

                        output_file.write(
                            chunk
                        )

                        bytes_written += len(
                            chunk
                        )

            if bytes_written <= 0:
                raise RuntimeError(
                    "Downloaded file is empty."
                )

            if (
                remote_size_bytes is not None
                and bytes_written != remote_size_bytes
            ):
                raise RuntimeError(
                    "Downloaded size does not match "
                    "Content-Length: "
                    f"{bytes_written} versus "
                    f"{remote_size_bytes}."
                )

            os.replace(
                temporary_path,
                destination_path,
            )

            return {
                "download_success": True,
                "http_status": 200,
                "remote_size_bytes": (
                    remote_size_bytes
                ),
                "local_size_bytes": int(
                    destination_path.stat().st_size
                ),
                "download_attempts": (
                    attempt_number
                ),
                "download_error": "",
            }

        except (
            HTTPError,
            URLError,
            TimeoutError,
            OSError,
            RuntimeError,
        ) as error:
            last_error = (
                f"{type(error).__name__}: {error}"
            )

            if temporary_path.exists():
                try:
                    temporary_path.unlink()

                except OSError:
                    pass

            if attempt_number < maximum_attempts:
                sleep_seconds = min(
                    2 ** attempt_number,
                    15,
                )

                print(
                    f"    Attempt {attempt_number} failed. "
                    f"Retrying in {sleep_seconds} seconds..."
                )

                time.sleep(
                    sleep_seconds
                )

    return {
        "download_success": False,
        "http_status": None,
        "remote_size_bytes": None,
        "local_size_bytes": (
            int(
                destination_path.stat().st_size
            )
            if destination_path.exists()
            else 0
        ),
        "download_attempts": (
            maximum_attempts
        ),
        "download_error": (
            last_error
        ),
    }


# ------------------------------------------------------------
# 9. Download and validate the 50 missing rasters
# ------------------------------------------------------------

download_results = []

total_records = len(
    recovery_manifest_df
)


print("\n" + "=" * 80)
print("DOWNLOADING MISSING S1HAND AND S2HAND RASTERS")
print("=" * 80)


for record_number, (
    _,
    manifest_row,
) in enumerate(
    recovery_manifest_df.iterrows(),
    start=1,
):
    scene_id = str(
        manifest_row[
            "scene_id"
        ]
    )

    component = str(
        manifest_row[
            "component"
        ]
    )

    public_url = str(
        manifest_row[
            "public_url"
        ]
    )

    local_path = Path(
        manifest_row[
            "local_path"
        ]
    )

    print(
        f"[{record_number:02d}/{total_records:02d}] "
        f"{scene_id} | {component}"
    )

    existing_valid = (
        existing_file_is_valid(
            local_path
        )
    )

    if existing_valid:
        print(
            "    Existing valid raster found. Skipping download."
        )

        download_result = {
            "download_success": True,
            "http_status": None,
            "remote_size_bytes": None,
            "local_size_bytes": int(
                local_path.stat().st_size
            ),
            "download_attempts": 0,
            "download_error": "",
        }

        download_status = (
            "SKIPPED_VALID_EXISTING"
        )

    else:
        if local_path.exists():
            try:
                local_path.unlink()

            except OSError:
                pass

        download_result = (
            download_file(
                url=public_url,
                destination_path=local_path,
            )
        )

        download_status = (
            "DOWNLOADED"
            if download_result[
                "download_success"
            ]
            else "FAILED"
        )

    raster_validation = (
        validate_geotiff(
            local_path
        )
        if local_path.exists()
        else {
            "raster_valid": False,
            "width": None,
            "height": None,
            "band_count": None,
            "dtype": None,
            "crs": None,
            "validation_error": (
                "Local file does not exist."
            ),
        }
    )

    validation_status = (
        "VALID"
        if (
            download_result[
                "download_success"
            ]
            and raster_validation[
                "raster_valid"
            ]
        )
        else "INVALID"
    )

    md5_value = ""

    if (
        local_path.exists()
        and raster_validation[
            "raster_valid"
        ]
    ):
        md5_value = (
            calculate_md5(
                local_path
            )
        )

    result_record = {
        **manifest_row.to_dict(),
        "download_status": (
            download_status
        ),
        "download_success": bool(
            download_result[
                "download_success"
            ]
        ),
        "http_status": (
            download_result[
                "http_status"
            ]
        ),
        "remote_size_bytes": (
            download_result[
                "remote_size_bytes"
            ]
        ),
        "local_exists": bool(
            local_path.exists()
        ),
        "local_size_bytes": (
            int(
                local_path.stat().st_size
            )
            if local_path.exists()
            else 0
        ),
        "download_attempts": int(
            download_result[
                "download_attempts"
            ]
        ),
        "download_error": (
            download_result[
                "download_error"
            ]
        ),
        "raster_valid": bool(
            raster_validation[
                "raster_valid"
            ]
        ),
        "raster_width": (
            raster_validation[
                "width"
            ]
        ),
        "raster_height": (
            raster_validation[
                "height"
            ]
        ),
        "raster_band_count": (
            raster_validation[
                "band_count"
            ]
        ),
        "raster_dtype": (
            raster_validation[
                "dtype"
            ]
        ),
        "raster_crs": (
            raster_validation[
                "crs"
            ]
        ),
        "raster_validation_error": (
            raster_validation[
                "validation_error"
            ]
        ),
        "md5": (
            md5_value
        ),
        "validation_status": (
            validation_status
        ),
    }

    download_results.append(
        result_record
    )

    print(
        f"    Status: {download_status} | "
        f"Raster: {validation_status} | "
        f"Size: "
        f"{result_record['local_size_bytes'] / (1024 ** 2):.2f} MB"
    )


recovery_results_df = pd.DataFrame(
    download_results
)


# ------------------------------------------------------------
# 10. Recovery validation
# ------------------------------------------------------------

component_validation_df = (
    recovery_results_df
    .groupby(
        "component",
        as_index=False,
    )
    .agg(
        expected_records=(
            "scene_id",
            "count",
        ),
        unique_scenes=(
            "scene_id",
            "nunique",
        ),
        downloaded_or_existing=(
            "download_success",
            "sum",
        ),
        valid_rasters=(
            "raster_valid",
            "sum",
        ),
        total_size_bytes=(
            "local_size_bytes",
            "sum",
        ),
    )
)


component_validation_df[
    "all_valid"
] = (
    component_validation_df[
        "valid_rasters"
    ]
    .eq(
        component_validation_df[
            "expected_records"
        ]
    )
)


scene_component_validation_df = (
    recovery_results_df
    .pivot_table(
        index="scene_id",
        columns="component",
        values="raster_valid",
        aggfunc="max",
        fill_value=False,
    )
    .reset_index()
)


for required_component in [
    "S1Hand",
    "S2Hand",
]:
    if (
        required_component
        not in scene_component_validation_df.columns
    ):
        scene_component_validation_df[
            required_component
        ] = False


scene_component_validation_df[
    "both_modalities_valid"
] = (
    scene_component_validation_df[
        "S1Hand"
    ].astype(bool)
    & scene_component_validation_df[
        "S2Hand"
    ].astype(bool)
)


recovery_validation = {
    "recovery_record_count_is_50": (
        len(
            recovery_results_df
        )
        == 50
    ),
    "recovery_scene_count_matches_required": (
        recovery_results_df[
            "scene_id"
        ].nunique()
        == len(required_scene_ids)
    ),
    "s1_record_count_matches_required": (
        (
            recovery_results_df[
                "component"
            ]
            == "S1Hand"
        ).sum()
        == len(required_scene_ids)
    ),
    "s2_record_count_matches_required": (
        (
            recovery_results_df[
                "component"
            ]
            == "S2Hand"
        ).sum()
        == len(required_scene_ids)
    ),
    "all_downloads_successful": (
        recovery_results_df[
            "download_success"
        ].all()
    ),
    "all_local_files_exist": (
        recovery_results_df[
            "local_exists"
        ].all()
    ),
    "all_rasters_valid": (
        recovery_results_df[
            "raster_valid"
        ].all()
    ),
    "all_scenes_have_both_modalities": (
        scene_component_validation_df[
            "both_modalities_valid"
        ].all()
    ),
}


print("\nRECOVERY VALIDATION")
print("-" * 80)

for validation_name, validation_result in (
    recovery_validation.items()
):
    print(
        f"{validation_name:<38}: "
        f"{bool(validation_result)}"
    )


# ------------------------------------------------------------
# 11. Save manifests and validation outputs
# ------------------------------------------------------------

RECOVERY_PLAN_CSV = (
    RECOVERY_OUTPUT_DIR
    / "missing_s1_s2_recovery_plan.csv"
)

RECOVERY_RESULTS_CSV = (
    RECOVERY_OUTPUT_DIR
    / "missing_s1_s2_recovery_results.csv"
)

COMPONENT_VALIDATION_CSV = (
    RECOVERY_OUTPUT_DIR
    / "missing_s1_s2_component_validation.csv"
)

SCENE_COMPONENT_VALIDATION_CSV = (
    RECOVERY_OUTPUT_DIR
    / "missing_s1_s2_scene_validation.csv"
)

RECOVERY_VALIDATION_CSV = (
    RECOVERY_OUTPUT_DIR
    / "missing_s1_s2_validation_checks.csv"
)

RECOVERY_SUMMARY_JSON = (
    RECOVERY_OUTPUT_DIR
    / "missing_s1_s2_recovery_summary.json"
)


recovery_manifest_df.to_csv(
    RECOVERY_PLAN_CSV,
    index=False,
)

recovery_results_df.to_csv(
    RECOVERY_RESULTS_CSV,
    index=False,
)

component_validation_df.to_csv(
    COMPONENT_VALIDATION_CSV,
    index=False,
)

scene_component_validation_df.to_csv(
    SCENE_COMPONENT_VALIDATION_CSV,
    index=False,
)


recovery_validation_df = pd.DataFrame(
    [
        {
            "validation_check": (
                validation_name
            ),
            "passed": bool(
                validation_result
            ),
        }
        for validation_name, validation_result in (
            recovery_validation.items()
        )
    ]
)


recovery_validation_df.to_csv(
    RECOVERY_VALIDATION_CSV,
    index=False,
)


recovery_generated_utc = (
    datetime.now(
        timezone.utc
    )
    .replace(
        microsecond=0
    )
    .isoformat()
)


recovery_summary = {
    "notebook": (
        "02A_recover_missing_s1_s2_downloads"
    ),
    "status": (
        "complete"
        if all(
            bool(
                value
            )
            for value in (
                recovery_validation.values()
            )
        )
        else "incomplete"
    ),
    "generated_utc": (
        recovery_generated_utc
    ),
    "required_scene_count": int(
        len(
            required_scene_ids
        )
    ),
    "recovery_record_count": int(
        len(
            recovery_results_df
        )
    ),
    "s1_valid_count": int(
        (
            recovery_results_df[
                "component"
            ].eq(
                "S1Hand"
            )
            & recovery_results_df[
                "raster_valid"
            ]
        ).sum()
    ),
    "s2_valid_count": int(
        (
            recovery_results_df[
                "component"
            ].eq(
                "S2Hand"
            )
            & recovery_results_df[
                "raster_valid"
            ]
        ).sum()
    ),
    "scene_with_both_modalities_count": int(
        scene_component_validation_df[
            "both_modalities_valid"
        ].sum()
    ),
    "total_downloaded_size_bytes": int(
        recovery_results_df[
            "local_size_bytes"
        ].sum()
    ),
    "validation_results": {
        validation_name: bool(
            validation_result
        )
        for validation_name, validation_result in (
            recovery_validation.items()
        )
    },
    "output_files": {
        "recovery_plan_csv": str(
            RECOVERY_PLAN_CSV
        ),
        "recovery_results_csv": str(
            RECOVERY_RESULTS_CSV
        ),
        "component_validation_csv": str(
            COMPONENT_VALIDATION_CSV
        ),
        "scene_validation_csv": str(
            SCENE_COMPONENT_VALIDATION_CSV
        ),
        "validation_checks_csv": str(
            RECOVERY_VALIDATION_CSV
        ),
    },
}


with open(
    RECOVERY_SUMMARY_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        recovery_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 12. Final report
# ------------------------------------------------------------

failed_recovery_df = (
    recovery_results_df[
        ~recovery_results_df[
            "raster_valid"
        ]
    ][
        [
            "scene_id",
            "component",
            "public_url",
            "local_path",
            "download_status",
            "download_error",
            "raster_validation_error",
        ]
    ]
)


print("\nCOMPONENT VALIDATION SUMMARY")
print("-" * 80)

display(
    component_validation_df
)


print("\n" + "=" * 80)
print("MISSING S1/S2 RECOVERY COMPLETE")
print("=" * 80)

print(
    f"Required scenes               : "
    f"{len(required_scene_ids):,}"
)

print(
    f"Recovery records              : "
    f"{len(recovery_results_df):,}"
)

print(
    f"Valid S1Hand rasters          : "
    f"{((recovery_results_df['component'] == 'S1Hand') & recovery_results_df['raster_valid']).sum():,}"
)

print(
    f"Valid S2Hand rasters          : "
    f"{((recovery_results_df['component'] == 'S2Hand') & recovery_results_df['raster_valid']).sum():,}"
)

print(
    f"Scenes with both modalities   : "
    f"{scene_component_validation_df['both_modalities_valid'].sum():,}"
)

print(
    f"Total local size              : "
    f"{recovery_results_df['local_size_bytes'].sum() / (1024 ** 2):,.2f} MB"
)

print(
    f"Overall recovery status       : "
    f"{recovery_summary['status']}"
)


print("\nOUTPUT FILES")
print("-" * 80)

for output_path in [
    RECOVERY_PLAN_CSV,
    RECOVERY_RESULTS_CSV,
    COMPONENT_VALIDATION_CSV,
    SCENE_COMPONENT_VALIDATION_CSV,
    RECOVERY_VALIDATION_CSV,
    RECOVERY_SUMMARY_JSON,
]:
    print(
        output_path
    )


if not failed_recovery_df.empty:
    print("\nFAILED OR INVALID RECOVERY RECORDS")
    print("-" * 80)

    display(
        failed_recovery_df
    )

    raise RuntimeError(
        f"{len(failed_recovery_df)} S1/S2 raster files "
        "failed download or GeoTIFF validation."
    )


print("\nRECOVERY PREVIEW")
print("-" * 80)

display(
    recovery_results_df[
        [
            "scene_id",
            "component",
            "download_status",
            "local_exists",
            "local_size_bytes",
            "raster_width",
            "raster_height",
            "raster_band_count",
            "raster_dtype",
            "raster_crs",
            "validation_status",
        ]
    ]
    .head(
        20
    )
)


print("\nNEXT STEP")
print("-" * 80)

print(
    "Return to Notebook 08 and rerun Cell 1 from the beginning. "
    f"It should now discover {len(required_scene_ids)} S1Hand and "
        f"{len(required_scene_ids)} S2Hand rasters."
)

print("=" * 80)

NOTEBOOK 02A: RECOVER MISSING S1HAND AND S2HAND RASTERS

DIRECTORIES
--------------------------------------------------------------------------------
Project root          : /home/adjeiowusu1/myproject/ResilientVLM
S1 output directory   : /home/adjeiowusu1/myproject/ResilientVLM/data/raw/sen1floods11/hand_labeled/S1Hand
S2 output directory   : /home/adjeiowusu1/myproject/ResilientVLM/data/raw/sen1floods11/hand_labeled/S2Hand
Recovery output       : /home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/recovery

SOURCE FILE VALIDATION
--------------------------------------------------------------------------------
vlm_scene_manifest        : True | /home/adjeiowusu1/myproject/ResilientVLM/data/processed/vlm_dataset/master/scene_multimodal_manifest.csv
scene_quality_manifest    : True | /home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/roadflood_vlm_scene_quality_manifest.csv
prototype_manifest        : True | /home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/roadflood_vlm_

    Status: SKIPPED_VALID_EXISTING | Raster: VALID | Size: 1.42 MB
[38/52] Spain_8565131 | S2Hand
    Existing valid raster found. Skipping download.
    Status: SKIPPED_VALID_EXISTING | Raster: VALID | Size: 2.18 MB
[39/52] Sri-Lanka_14484 | S1Hand
    Existing valid raster found. Skipping download.
    Status: SKIPPED_VALID_EXISTING | Raster: VALID | Size: 1.72 MB
[40/52] Sri-Lanka_14484 | S2Hand
    Existing valid raster found. Skipping download.
    Status: SKIPPED_VALID_EXISTING | Raster: VALID | Size: 2.44 MB
[41/52] Sri-Lanka_92824 | S1Hand
    Existing valid raster found. Skipping download.
    Status: SKIPPED_VALID_EXISTING | Raster: VALID | Size: 1.71 MB
[42/52] Sri-Lanka_92824 | S2Hand
    Existing valid raster found. Skipping download.
    Status: SKIPPED_VALID_EXISTING | Raster: VALID | Size: 2.28 MB
[43/52] USA_1068362 | S1Hand
    Existing valid raster found. Skipping download.
    Status: SKIPPED_VALID_EXISTING | Raster: VALID | Size: 1.40 MB
[44/52] USA_1068362 | S2Han

,component,expected_records,unique_scenes,downloaded_or_existing,valid_rasters,total_size_bytes,all_valid
0,S1Hand,26,26,26,26,42849822,True
1,S2Hand,26,26,26,26,61674122,True



MISSING S1/S2 RECOVERY COMPLETE
Required scenes               : 26
Recovery records              : 52
Valid S1Hand rasters          : 26
Valid S2Hand rasters          : 26
Scenes with both modalities   : 26
Total local size              : 99.68 MB
Overall recovery status       : incomplete

OUTPUT FILES
--------------------------------------------------------------------------------
/home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/recovery/missing_s1_s2_recovery_plan.csv
/home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/recovery/missing_s1_s2_recovery_results.csv
/home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/recovery/missing_s1_s2_component_validation.csv
/home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/recovery/missing_s1_s2_scene_validation.csv
/home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/recovery/missing_s1_s2_validation_checks.csv
/home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/recovery/missing_s1_s2_recovery_summary.json

RECOVERY 

,scene_id,component,download_status,local_exists,local_size_bytes,raster_width,raster_height,raster_band_count,raster_dtype,raster_crs,validation_status
0,Ghana_141910,S1Hand,SKIPPED_VALID_EXISTING,True,1796026,512,512,2,"float32,float32",EPSG:4326,VALID
1,Ghana_141910,S2Hand,SKIPPED_VALID_EXISTING,True,2287617,512,512,13,"int16,int16,int16,int16,int16,int16,int16,int1...",EPSG:4326,VALID
2,India_1018327,S1Hand,SKIPPED_VALID_EXISTING,True,1670156,512,512,2,"float32,float32",EPSG:4326,VALID
3,India_1018327,S2Hand,SKIPPED_VALID_EXISTING,True,2334123,512,512,13,"int16,int16,int16,int16,int16,int16,int16,int1...",EPSG:4326,VALID
4,India_1050276,S1Hand,SKIPPED_VALID_EXISTING,True,1668657,512,512,2,"float32,float32",EPSG:4326,VALID
5,India_1050276,S2Hand,SKIPPED_VALID_EXISTING,True,2321695,512,512,13,"int16,int16,int16,int16,int16,int16,int16,int1...",EPSG:4326,VALID
6,India_1068117,S1Hand,SKIPPED_VALID_EXISTING,True,1671798,512,512,2,"float32,float32",EPSG:4326,VALID
7,India_1068117,S2Hand,SKIPPED_VALID_EXISTING,True,2450753,512,512,13,"int16,int16,int16,int16,int16,int16,int16,int1...",EPSG:4326,VALID
8,India_285297,S1Hand,SKIPPED_VALID_EXISTING,True,1669481,512,512,2,"float32,float32",EPSG:4326,VALID
9,India_285297,S2Hand,SKIPPED_VALID_EXISTING,True,2398643,512,512,13,"int16,int16,int16,int16,int16,int16,int16,int1...",EPSG:4326,VALID



NEXT STEP
--------------------------------------------------------------------------------
Return to Notebook 08 and rerun Cell 1 from the beginning. It should now discover 26 S1Hand and 26 S2Hand rasters.
